In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.utils import pmf_utils

### Config

In [3]:
from config import dir_config, main_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

# Following viewing times to be added in reaction time to account for actual accumulation time
min_viewing_time_uw = 0.3
min_viewing_time_ucla = 0.3

acceptance_criteria = main_config.acceptance_criteria

### Utils functions

### Load data

In [4]:
aggregate_data = pd.read_csv(Path(raw_dir, "aggregate_data.csv"), index_col=None)
aggregate_data.shape

(134537, 22)

In [5]:
sessions = aggregate_data.groupby(["subject_id", "session_id"]).size().reset_index().rename(columns={0: "count"})
sessions

,subject_id,session_id,count
0,DBM11,10_1,624
1,DBM11,11_1,499
2,DBM11,12_1,611
3,DBM11,13_1,534
4,DBM11,14_1,555
...,...,...,...
293,PSUIM9,5_1,302
294,PSUIM9,6_1,418
295,PSUIM9,7_1,463
296,PSUIM9,8_1,375


#### Add trial-type to column with following mapping
- -3: 'late_left_pulse'
- -2: 'middle_left_pulse'
- -1: 'early_left_pulse'
- 0: 'control'
- 1: 'early_right_pulse'
- 2: 'middle_right_pulse'
- 3: 'late_right_pulse'


In [6]:
# based on timing divide pulse_onset trials into 3 categories
pulse_type = np.zeros(len(aggregate_data))
pulse_type[(aggregate_data.pulse_onset >= 0) & (aggregate_data.pulse_onset < 0.16)] = 1
pulse_type[(aggregate_data.pulse_onset >= 0.16) & (aggregate_data.pulse_onset < 0.32)] = 2
pulse_type[(aggregate_data.pulse_onset >= 0.32) & (aggregate_data.pulse_onset < 0.48)] = 3

pulse_type[aggregate_data.signed_coherence == aggregate_data.pulse_coherence] = 0
pulse_type[aggregate_data.is_pulse_trial == False] = 0

pulse_type[aggregate_data.reaction_time < (aggregate_data.pulse_onset + aggregate_data.pulse_duration)] = 0  # remove trials where the response was made before the pulse onset

pulse_type = (pulse_type * np.sign(aggregate_data.pulse_coherence)).fillna(0).astype(int)
aggregate_data["pulse_type"] = pulse_type

In [7]:
# Function to validate assertions and print errors if conditions fail
def check_assertion(df, condition, message) -> None:
    failing_cases = df.loc[condition.index[~condition]]
    if not failing_cases.empty:
        pass
    assert failing_cases.empty, message


# Check conditions for pulse_type 1
subdata = aggregate_data[aggregate_data["pulse_type"] == 1]
condition = (subdata["pulse_onset"] >= 0) & (subdata["pulse_onset"] < 0.16)
check_assertion(subdata, condition, "pulse_onset should be between 0 and 0.16 for pulse_type 1")
# Check conditions for pulse_type 2
subdata = aggregate_data[aggregate_data["pulse_type"] == 2]
condition = (subdata["pulse_onset"] >= 0.16) & (subdata["pulse_onset"] < 0.32)
check_assertion(subdata, condition, "pulse_onset should be between 0.16 and 0.32 for pulse_type 2")
# Check conditions for pulse_type 3
subdata = aggregate_data[aggregate_data["pulse_type"] == 31]
condition = (subdata["pulse_onset"] >= 0.32) & (subdata["pulse_onset"] < 0.48)
check_assertion(subdata, condition, "pulse_onset should be between 0.32 and 0.48 for pulse_type 3")
# Check conditions for pulse_type -1
subdata = aggregate_data[aggregate_data["pulse_type"] == -1]
condition = (subdata["pulse_onset"] >= 0) & (subdata["pulse_onset"] < 0.16)
check_assertion(subdata, condition, "pulse_onset should be between 0 and 0.16 for pulse_type -1")
# Check conditions for pulse_type -2
subdata = aggregate_data[aggregate_data["pulse_type"] == -2]
condition = (subdata["pulse_onset"] >= 0.16) & (subdata["pulse_onset"] < 0.32)
check_assertion(subdata, condition, "pulse_onset should be between 0.16 and 0.32 for pulse_type -2")
# Check conditions for pulse_type -3
subdata = aggregate_data[aggregate_data["pulse_type"] == -31]
condition = (subdata["pulse_onset"] >= 0.32) & (subdata["pulse_onset"] < 0.48)
check_assertion(subdata, condition, "pulse_onset should be between 0.32 and 0.48 for pulse_type -3")

### Sessions and Trials to remove

In [8]:
valid_data = pd.DataFrame()

# get the unique subject ids and session ids combinations
sessions = aggregate_data.groupby(["subject_id", "session_id"]).size().reset_index().rename(columns={0: "count"})

for _idx, session in sessions.iterrows():
    session_data = aggregate_data[(aggregate_data["subject_id"] == session["subject_id"]) & (aggregate_data["session_id"] == session["session_id"])]
    session_data = session_data[session_data["is_correction_trial"] != 1]
    session_data = session_data[(session_data.reaction_time >= acceptance_criteria.min_reaction_time) & (session_data.reaction_time <= acceptance_criteria.max_reaction_time)]

    if session_data.shape[0] < acceptance_criteria["min_trials"]:
        continue

    control_data = session_data[session_data["pulse_type"] == 0]
    if control_data.shape[0]:
        x, y, model, x_hat, y_hat = pmf_utils.get_psychometric_data(control_data, model_type=main_config.psych_model, x_model_range=[-100, 100])
        if model.coefs_["lapse_rate"] < acceptance_criteria.max_lapse_rate:
            if model.coefs_["guess_rate"] < acceptance_criteria.max_guess_rate:
                if abs(model.coefs_["mean"]) < acceptance_criteria.max_abs_alpha:
                    if model.coefs_["var"] < acceptance_criteria.max_beta:
                        valid_data = pd.concat([valid_data, session_data])
    else:
        pass


print(f"Total subjects: {valid_data['subject_id'].nunique()}, Total sessions: {valid_data['session_id'].nunique()}, Total trials: {len(valid_data)}")
valid_data.groupby("subject_id").agg(n_sessions=("session_id", "nunique"), n_rows=("session_id", "size")).reset_index()

Total subjects: 10, Total sessions: 26, Total trials: 25508


,subject_id,n_sessions,n_rows
0,DBM12,2,1700
1,DBM13,1,719
2,DBM14,1,829
3,DBM15,3,1095
4,DBM5,5,2646
5,DBM6,6,3217
6,PSUIM10,1,425
7,PSUIM5,5,2185
8,PSUIM6,15,5845
9,PSUIM7,11,6847


In [9]:
# map choices to 0 and 1
valid_data["choice"] = valid_data["choice"].map({1: 1, -1: 0})
aggregate_data["choice"] = aggregate_data["choice"].map({1: 1, -1: 0})

In [10]:
# create new column agg_code as {idx_valid_trial}_{subject_id}_{session_id}
valid_data["agg_code"] = valid_data.apply(lambda row: f"{row['subject_id']}_{row['session_id']}_{row['idx_valid_trial']}", axis=1)


# Reorder columns to make agg_code the first one
cols = ["agg_code"] + [col for col in valid_data.columns if col != "agg_code"]
valid_data = valid_data[cols]

In [11]:
aggregate_data.to_csv(Path(processed_dir, "aggregate_data.csv"), index=False)

valid_data.to_csv(Path(processed_dir, "valid_data.csv"), index=False)